<a href="https://colab.research.google.com/github/Vishu235/MetaBEARS/blob/main/colab/MetaBEARS_BDD_OIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MetaBEARS — BDD-OIA Practical Baseline Runbook

This notebook establishes the BDD-OIA practical baseline and reusable data artifacts needed before adapting the MetaBEARS diagnostic layer to multi-label driving actions and 21 labelled concepts.

## Before running

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Upload the official `lastframe.zip` to:

`MyDrive/PES - Semester 4/bears_data/lastframe.zip`

This is a practical reconstruction using ResNet50 ImageNet features. It is not an exact reproduction of the paper's unavailable Faster-RCNN/CBM-AUC `bdd_2048.zip` feature setup. Run the smoke cells before enabling full preprocessing or training.

In [ ]:
#@title 1. Configuration
REPO_URL = "https://github.com/Vishu235/MetaBEARS.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/MetaBEARS"  #@param {type:"string"}

DRIVE_DATA_DIR = "/content/drive/MyDrive/PES - Semester 4/bears_data"  #@param {type:"string"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/PES - Semester 4/metabears_bdd_oia"  #@param {type:"string"}
LASTFRAME_ZIP = f"{DRIVE_DATA_DIR}/lastframe.zip"
BDD_DATA_REL = "data/bdd2048_resnet"

RUN_SMOKE = True  #@param {type:"boolean"}
RESTORE_PREPROCESSED = True  #@param {type:"boolean"}
RUN_FULL_PREPROCESS = False  #@param {type:"boolean"}
RUN_FULL_TRAINING = False  #@param {type:"boolean"}
RUN_NO_ENTROPY = True  #@param {type:"boolean"}
RUN_ENTROPY = True  #@param {type:"boolean"}
RUN_CONCEPT_SUPERVISION = False  #@param {type:"boolean"}

BDD_EPOCHS = 30  #@param {type:"integer"}
BDD_BATCH_SIZE = 256  #@param {type:"integer"}
FEATURE_BATCH_SIZE = 64  #@param {type:"integer"}
FEATURE_WORKERS = 4  #@param {type:"integer"}
SEEDS = [0, 10, 20]

print('Configuration ready. Full training enabled:', RUN_FULL_TRAINING)


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Clone or update MetaBEARS
import os
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
if repo.exists() and (repo / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin'], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=repo, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=repo, check=True)
elif repo.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)


In [ ]:
#@title 4. Install Colab-safe dependencies
import subprocess

subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements.colab.txt'], cwd=REPO_DIR, check=True)
print('Dependencies installed without replacing Colab CUDA PyTorch.')


In [ ]:
#@title 5. Diagnostics and input validation
import hashlib
from pathlib import Path

subprocess.run(['python', 'colab_runner.py', '--job', 'diagnostics'], cwd=REPO_DIR, check=True)
lastframe = Path(LASTFRAME_ZIP)
if not lastframe.exists():
    raise FileNotFoundError(f'Upload lastframe.zip to {lastframe}')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

Path(DRIVE_RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print('lastframe.zip size (GB):', round(lastframe.stat().st_size / 1024**3, 3))
print('lastframe.zip SHA-256:', sha256(lastframe))


In [ ]:
#@title 6. Reusable BDD job helper
import shlex
import subprocess
from pathlib import Path

def run_job(job, extra_args=None):
    command = ['python', 'colab_runner.py', '--job', job, '--lastframe-zip', LASTFRAME_ZIP]
    if extra_args:
        command.extend(str(value) for value in extra_args)
    print('\n>>>', ' '.join(shlex.quote(part) for part in command), flush=True)
    completed = subprocess.run(command, cwd=REPO_DIR, check=False)
    if completed.returncode:
        logs = sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda p: p.stat().st_mtime)
        if logs:
            print(''.join(logs[-1].read_text(errors='replace').splitlines(True)[-180:]))
        raise SystemExit(f'{job} failed with exit code {completed.returncode}')
    logs = sorted((Path(REPO_DIR) / 'logs').glob('*.log'), key=lambda p: p.stat().st_mtime)
    return logs[-1] if logs else None

print('Job helper ready.')


In [ ]:
#@title 7. Restore reusable full ResNet50 features from Drive
import shutil
from pathlib import Path

local_features = Path(REPO_DIR) / 'BDD_OIA' / BDD_DATA_REL
# A single zip transfers over the Drive FUSE mount in seconds; copying tens
# of thousands of individual .pt files the old way (shutil.copytree on a
# directory) can take 20+ minutes because each tiny file is a separate
# network round trip through Drive's API.
drive_feature_archive = Path(DRIVE_RESULTS_DIR) / 'preprocessed' / 'bdd2048_resnet.zip'
drive_complete = drive_feature_archive.exists()
if RESTORE_PREPROCESSED and drive_complete and not local_features.exists():
    local_features.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(drive_feature_archive), str(local_features))
    print('Restored preprocessed features from archive:', drive_feature_archive, '->', local_features)
else:
    print('Local features exist:', local_features.exists())
    print('Drive feature archive exists:', drive_complete)


In [ ]:
#@title 8. BDD-OIA smoke preprocessing and training
if RUN_SMOKE:
    smoke_data = 'data/bdd2048_colab_smoke'
    run_job('bdd_preprocess_smoke', [
        '--bdd-output', smoke_data, '--limit-per-split', '8',
        '--feature-weights', 'none', '--feature-batch-size', '8',
    ])
    smoke_log = run_job('bdd_train_smoke', [
        '--bdd-output', smoke_data, '--seed', '0',
        '--bdd-model-name', 'dpl_auc',
    ])
    smoke_dir = Path(DRIVE_RESULTS_DIR) / 'smoke'
    smoke_dir.mkdir(parents=True, exist_ok=True)
    if smoke_log is not None:
        shutil.copy2(smoke_log, smoke_dir / smoke_log.name)
    print('BDD smoke workflow passed.')
else:
    print('BDD smoke workflow disabled.')

In [ ]:
#@title 9. Full BDD-OIA ResNet50 preprocessing
preprocess_marker = local_features / 'preprocess_summary.json'
if preprocess_marker.exists():
    print('Full feature directory already complete; preprocessing skipped:', local_features)
elif RUN_FULL_PREPROCESS:
    if local_features.exists():
        print('Found an incomplete feature directory; re-running with --force:', local_features)
    run_job('bdd_preprocess_full', [
        '--bdd-output', BDD_DATA_REL, '--feature-weights', 'imagenet',
        '--feature-batch-size', str(FEATURE_BATCH_SIZE),
        '--feature-workers', str(FEATURE_WORKERS),
    ])
    if not preprocess_marker.exists():
        raise FileNotFoundError(
            f'Preprocessing finished without producing {preprocess_marker}; inspect the job log before training.'
        )
    # Zip locally (fast, local disk to local disk) and copy ONE file to
    # Drive, instead of shutil.copytree-ing ~70k tiny per-image .pt files
    # through the slow Drive FUSE mount.
    drive_feature_archive.parent.mkdir(parents=True, exist_ok=True)
    archive_base = Path(REPO_DIR) / 'BDD_OIA' / 'bdd2048_resnet_features'
    archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=str(local_features)))
    shutil.copy2(archive_path, drive_feature_archive)
    archive_path.unlink()
    print('Persisted reusable features archive to:', drive_feature_archive)
else:
    print('Full preprocessing disabled or incomplete. Enable RUN_FULL_PREPROCESS if no complete reusable feature directory exists.')


In [ ]:
#@title 10. Full multi-seed BDD-OIA baseline training
import json
from datetime import datetime, timezone

variants = []
if RUN_NO_ENTROPY:
    variants.append(('base', 'dpl_auc', 0.0, 0.0))
if RUN_ENTROPY:
    variants.append(('entropy', 'dpl_auc', 1.0, 0.0))
if RUN_CONCEPT_SUPERVISION:
    variants.append(('concept_supervision', 'dpl_auc', 0.0, 1.0))

records = []
if RUN_FULL_TRAINING:
    if not (local_features / 'preprocess_summary.json').exists():
        raise FileNotFoundError(
            f'Full BDD features are missing or incomplete at {local_features} '
            '(no preprocess_summary.json). Run full preprocessing to completion first.'
        )
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
    for seed in SEEDS:
        for condition, model_name, entropy_weight, concept_weight in variants:
            suffix = ('_csup' if concept_weight > 0 else '') + ('_entropy' if entropy_weight > 0 else '')
            run_name = f'{model_name}{suffix}-{seed}'
            seed_dir = Path(DRIVE_RESULTS_DIR) / 'runs' / run_name
            completion_path = seed_dir / 'completed_run.json'
            if completion_path.is_file():
                completed = json.loads(completion_path.read_text())
                completed['reused'] = True
                records.append(completed)
                print('Reusing completed BDD run:', run_name)
                continue
            log_path = run_job('bdd_train_full', [
                '--bdd-output', BDD_DATA_REL, '--epochs', str(BDD_EPOCHS),
                '--bdd-batch-size', str(BDD_BATCH_SIZE), '--seed', str(seed),
                '--bdd-model-name', model_name, '--w-entropy', str(entropy_weight),
                '--h-labeled-param', str(concept_weight),
            ])
            seed_dir.mkdir(parents=True, exist_ok=True)
            artifact_pairs = [
                (Path(REPO_DIR) / 'BDD_OIA' / 'out' / 'bdd' / run_name, seed_dir / 'out'),
                (Path(REPO_DIR) / 'BDD_OIA' / 'models' / 'bdd' / run_name, seed_dir / 'model'),
            ]
            for source, destination in artifact_pairs:
                if source.exists():
                    shutil.copytree(source, destination, dirs_exist_ok=True)
            if log_path is not None:
                shutil.copy2(log_path, seed_dir / log_path.name)
            test_csv = seed_dir / 'out' / 'test_results_of_BDD.csv'
            model_files = list((seed_dir / 'model').glob('model_best-*.pth.tar'))
            if not test_csv.is_file() or not model_files:
                raise FileNotFoundError(f'Incomplete persisted BDD run: {run_name}')
            record = {
                'condition': condition, 'seed': seed, 'model_name': model_name,
                'entropy_weight': entropy_weight, 'concept_weight': concept_weight,
                'run_name': run_name, 'reused': False,
                'test_results': str(test_csv),
            }
            completion_path.write_text(json.dumps(record, indent=2))
            records.append(record)

    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'git_commit': commit, 'dataset': 'BDD-OIA practical reconstruction',
        'feature_source': 'ResNet50 ImageNet features from lastframe.zip',
        'lastframe_sha256': sha256(Path(LASTFRAME_ZIP)),
        'epochs': BDD_EPOCHS, 'batch_size': BDD_BATCH_SIZE,
        'seeds': SEEDS, 'runs': records,
    }
    manifest_path = Path(DRIVE_RESULTS_DIR) / 'baseline_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print('Full BDD training complete. Manifest:', manifest_path)
else:
    print('Full training disabled. Run the smoke workflow first.')

In [ ]:
#@title 11. Build a compact BDD result summary
import csv
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

def binary_ece(y_true, probability, bins=10):
    y_true = np.asarray(y_true).astype(int)
    probability = np.clip(np.asarray(probability, dtype=float), 0.0, 1.0)
    prediction = (probability >= 0.5).astype(int)
    confidence = np.where(prediction == 1, probability, 1.0 - probability)
    correct = (prediction == y_true).astype(float)
    result = 0.0
    edges = np.linspace(0.0, 1.0, bins + 1)
    for index in range(bins):
        mask = (confidence >= edges[index]) & (confidence <= edges[index + 1]) if index == 0 else (confidence > edges[index]) & (confidence <= edges[index + 1])
        if mask.any():
            result += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(result)

rows = []
csv_paths = set((Path(REPO_DIR) / 'BDD_OIA' / 'out' / 'bdd').glob('*/test_results_of_BDD.csv'))
csv_paths.update(Path(DRIVE_RESULTS_DIR).glob('runs/*/out/test_results_of_BDD.csv'))
for csv_path in sorted(csv_paths):
    parsed = []
    with csv_path.open() as handle:
        for row in csv.reader(handle):
            values = [float(value) for value in row if value != '']
            if values:
                parsed.append(values)
    if not parsed:
        continue
    array = np.asarray(parsed, dtype=float)
    y_true = array[:, 0:4]
    y_prob = array[:, 5:13][:, [1, 3, 5, 7]]
    c_prob = array[:, 13:34]
    c_true = array[:, -21:]
    run_name = csv_path.parent.parent.name if csv_path.parent.name == 'out' else csv_path.parent.name
    rows.append({
        'run': run_name,
        'action_macro_f1': f1_score(y_true, y_prob >= 0.5, average='macro', zero_division=0),
        'concept_macro_f1': f1_score(c_true, c_prob >= 0.5, average='macro', zero_division=0),
        'action_mean_ece': float(np.mean([binary_ece(y_true[:, i], y_prob[:, i]) for i in range(4)])),
        'concept_mean_ece': float(np.mean([binary_ece(c_true[:, i], c_prob[:, i]) for i in range(21)])),
        'action_exact_match': accuracy_score(y_true, y_prob >= 0.5),
        'concept_exact_match': accuracy_score(c_true, c_prob >= 0.5),
    })

summary = pd.DataFrame(rows)
display(summary)
summary_path = Path(DRIVE_RESULTS_DIR) / 'bdd_results_summary.csv'
summary.to_csv(summary_path, index=False)
print('Summary saved to:', summary_path)


In [ ]:
#@title 12. Archive repository-side outputs to Drive
run_job('archive_results')
archives = sorted((Path(REPO_DIR) / 'colab_outputs').glob('bears_results_*.zip'), key=lambda p: p.stat().st_mtime)
if not archives:
    raise FileNotFoundError('No result archive was created.')
archive_target = Path(DRIVE_RESULTS_DIR) / archives[-1].name
shutil.copy2(archives[-1], archive_target)
print('Result archive copied to:', archive_target)


## MetaBEARS diagnostic evaluation

The cells above only produce the practical BDD-OIA baseline (three training
variants, three seeds each). The cells below pull the BDD-specific MetaBEARS
adapter (`metacog/bdd.py`, `metacog/bdd_runner.py`) and evaluate a same-variant
checkpoint ensemble with it. Checkpoints are read from
`metabears_bdd_oia/runs/<run_name>/model/` in Drive — the same place the
training cells above persist to — not from local Colab disk, since a same-seed
checkpoint set may have been trained across multiple sessions or on a
different machine entirely.</cell id="cell-13">


In [ ]:
#@title 13. Pull latest MetaBEARS code
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'log', '--oneline', '-3'], cwd=REPO_DIR, check=True)

In [ ]:
#@title 14. Locate BDD-OIA MetaBEARS checkpoint ensemble
BDD_METABEARS_VARIANT_SUFFIX = ""  #@param ["", "_entropy", "_csup"] {allow-input: true}
BDD_METABEARS_SEEDS = [0, 10, 20]  #@param

bdd_checkpoint_paths = []
bdd_checkpoint_missing = []
for seed in BDD_METABEARS_SEEDS:
    run_name = f"dpl_auc{BDD_METABEARS_VARIANT_SUFFIX}-{seed}"
    candidate = (
        Path(DRIVE_RESULTS_DIR) / "runs" / run_name / "model" / f"model_best-{seed}.pth.tar"
    )
    if candidate.is_file():
        bdd_checkpoint_paths.append(candidate)
    else:
        bdd_checkpoint_missing.append(str(candidate))

if bdd_checkpoint_missing:
    raise FileNotFoundError(
        "Missing BDD-OIA checkpoints in Drive:\n" + "\n".join(bdd_checkpoint_missing)
    )
if not local_features.exists():
    raise FileNotFoundError(
        f"{local_features} is missing locally. Run cell 7 (restore) or cell 9 "
        "(full preprocessing) first so the MetaBEARS evaluation has data to read."
    )

print(f"Using variant 'dpl_auc{BDD_METABEARS_VARIANT_SUFFIX}' with {len(bdd_checkpoint_paths)} members:")
for path in bdd_checkpoint_paths:
    print(" -", path)

In [ ]:
#@title 15. Run BDD-OIA MetaBEARS evaluation
bdd_metabears_output = (
    Path(DRIVE_RESULTS_DIR) / "metabears" / f"dpl_auc{BDD_METABEARS_VARIANT_SUFFIX}"
)
run_job("metabears_bdd", [
    "--bdd-metabears-data-dir", str(local_features),
    "--bdd-metabears-output-dir", str(bdd_metabears_output),
    "--bdd-metabears-checkpoints", *[str(path) for path in bdd_checkpoint_paths],
])
print("BDD-OIA MetaBEARS artifacts:", bdd_metabears_output)

summary_path = bdd_metabears_output / "run_summary.json"
if summary_path.is_file():
    import json as _json
    payload = _json.loads(summary_path.read_text())
    print(_json.dumps(payload.get("splits", {}), indent=2, sort_keys=True))

## Completion gate

After the smoke workflow and practical baselines succeed, share `baseline_manifest.json`, `bdd_results_summary.csv`, and the result archive. The next implementation stage is a BDD-specific member-preserving MetaBEARS adapter for multi-label actions and binary concept distributions. These baseline results must not yet be presented as MetaBEARS shortcut-detection results.
